In [ ]:
from google.colab import drive
import torch
import torchvision
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import os
import random
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:

drive.mount('/content/drive')

Mounted at /content/drive


**Define Augmentation and Transforms for the training set**

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.GaussianBlur(kernel_size=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

**Define Transforms for validation and test sets**

In [ ]:
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

**Define the datasets path**

In [ ]:
train_path = '/content/drive/MyDrive/Flip_Prediction/images/training'
test_path  = '/content/drive/MyDrive/Flip_Prediction/images/testing'

print("Train folder contents:", os.listdir(train_path))
print("Test folder contents:", os.listdir(test_path))

Train folder contents: ['notflip', 'flip']
Test folder contents: ['notflip', 'flip']


**Load datasets with transforms**

In [ ]:
# Load training dataset with augmentation
train_dataset = datasets.ImageFolder(
    root=train_path,
    transform=train_transform   # training transforms
)

# Load testing dataset without augmentation
test_dataset = datasets.ImageFolder(
    root=test_path,
    transform=val_test_transform  # clean validation/test transforms
)

**Split the training set into: training set= 90%, validation set = 10% using stratified split method**


In [ ]:
full_dataset = train_dataset

# Get all labels from the full dataset
labels = [sample[1] for sample in full_dataset.imgs]

# Get all indices of the dataset
indices = list(range(len(full_dataset)))

# Generate 1 random seed
NUM_SEEDS = 1
seeds = [random.randint(1000, 9999) for _ in range(NUM_SEEDS)]
seed_to_use = seeds[0]

print("Using random seed:", seed_to_use)

# Stratified split: 10% for validation
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.1,
    stratify=labels,
    random_state=seed_to_use
)

# Create subsets for train and validation from full_dataset
train_dataset = Subset(full_dataset, train_idx)
val_dataset   = Subset(full_dataset, val_idx)

# Apply transforms
train_dataset.dataset.transform = train_transform      # training with augmentation
val_dataset.dataset.transform   = val_test_transform   # clean validation

# Print sizes to confirm
print("Number of training images:", len(train_dataset))
print("Number of validation images:", len(val_dataset))

Using random seed: 2199
Number of training images: 2152
Number of validation images: 240


**Create DataLoaders**

In [ ]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Verify one batch
images, labels = next(iter(train_loader))
print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)

Batch images shape: torch.Size([32, 3, 224, 224])
Batch labels shape: torch.Size([32])


**Load the model**

In [ ]:
# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load pretrained ResNet50
model = models.resnet50(pretrained=True)
print("model loaded")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s]


model loaded


**Set up the model for learning**

**Replace Final Classifier (2 Classes)**

In [ ]:
num_features = model.fc.in_features

model.fc = nn.Linear(num_features, 2)

model = model.to(device)

print("Classifier replaced")

Classifier replaced


**Freeze Backbone (Feature Extractor Mode)**

In [ ]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze classifier only
for param in model.fc.parameters():
    param.requires_grad = True

print("Backbone frozen, classifier trainable")

Backbone frozen, classifier trainable


**Define Loss Function and Optimizer**

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("Loss and optimizer ready")

Loss and optimizer ready


**Set hyperparameters**

In [ ]:
num_epochs = 10
batch_size = 32
learning_rate = 0.001

**Train the model using training set**

In [ ]:

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    for inputs, labels in tqdm(train_loader):

        inputs = inputs.to(device)
        labels = labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item() * inputs.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Training Loss: {epoch_train_loss:.4f}")

100%|██████████| 68/68 [13:56<00:00, 12.31s/it]



Epoch [1/10]
Training Loss: 0.5907


100%|██████████| 68/68 [01:03<00:00,  1.07it/s]



Epoch [2/10]
Training Loss: 0.3877


100%|██████████| 68/68 [01:03<00:00,  1.07it/s]


...


Epoch [8/10]
Training Loss: 0.1767


100%|██████████| 68/68 [01:03<00:00,  1.06it/s]



Epoch [9/10]
Training Loss: 0.1689


100%|██████████| 68/68 [01:03<00:00,  1.07it/s]


Epoch [10/10]
Training Loss: 0.1694


**Validate the trained model using validation set**

In [ ]:
model.eval()

val_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for inputs, labels in val_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        val_loss += loss.item() * inputs.size(0)

        # Get predicted class
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

val_loss = val_loss / len(val_loader.dataset)
val_acc = correct / total

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

Validation Loss: 0.1357
Validation Accuracy: 0.9750


**Test the validated model using test set**

In [ ]:
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for inputs, labels in test_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss = test_loss / len(test_loader.dataset)
test_accuracy = correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.1687
Test Accuracy: 0.9581


**Evaluate the model using classification report**

**Define the evaluation function**

In [ ]:
def evaluate_model(model, dataloader, class_names, dataset_name="Dataset"):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # ---------- Classification Report ----------
    print(f"\n===== {dataset_name} Classification Report =====")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # ---------- Confusion Matrix (TEXT ONLY) ----------
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n===== {dataset_name} Confusion Matrix =====")
    print(cm)

**Define the classes**

In [ ]:
class_names = ['flip', 'notflip']

**Evaluate the model**

In [ ]:
evaluate_model(model, train_loader, class_names, dataset_name="Training Set")
evaluate_model(model, val_loader, class_names, dataset_name="Validation Set")
evaluate_model(model, test_loader, class_names, dataset_name="Test Set")


===== Training Set Classification Report =====
              precision    recall  f1-score   support

        flip       0.95      0.98      0.97      1045
     notflip       0.98      0.95      0.97      1107

    accuracy                           0.97      2152
   macro avg       0.97      0.97      0.97      2152
weighted avg       0.97      0.97      0.97      2152


===== Training Set Confusion Matrix =====
[[1028   17]
 [  53 1054]]

===== Validation Set Classification Report =====
              precision    recall  f1-score   support

        flip       0.97      0.97      0.97       117
     notflip       0.98      0.98      0.98       123

    accuracy                           0.97       240
   macro avg       0.97      0.97      0.97       240
weighted avg       0.97      0.97      0.97       240


===== Validation Set Confusion Matrix =====
[[114   3]
 [  3 120]]

===== Test Set Classification Report =====
              precision    recall  f1-score   support

        fli

**Save the Model Weights**

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Flip_Prediction/Trained_models/Trained_ResNet_model.pth'
torch.save(model.state_dict(), MODEL_PATH)

print("Model saved successfully!")

Model saved successfully!
